In [ ]:
!pip install langchain openai pymilvus pdfplumber beautifulsoup4


In [1]:
import json
import os
from typing import List, Dict

import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI 
from dotenv import load_dotenv

In [2]:
load_dotenv()

# Retrieve API keys from environment variables
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError("OpenAI API key not found. Please set OPENAI_API_KEY in .env file.")


In [3]:
MODEL="gpt-4o-mini"
client = OpenAI()

In [ ]:
class RAGApplication:
    def __init__(self, json_path: str, collection_name: str = 'datasources/knowledge_base'):
        """
        Initialize the RAG application with a JSON file and ChromaDB collection
        
        :param json_path: Path to the JSON file containing knowledge base
        :param collection_name: Name of the ChromaDB collection
        """
        # Initialize ChromaDB client with specific folder and index

             
        # Use OpenAI's embedding function
        self.embedding_function = embedding_functions.OpenAIEmbeddingFunction(
            api_key=OPENAI_API_KEY,
            model_name="text-embedding-ada-002"
        )

        self.chroma_client = chromadb.PersistentClient(path="./chromadb")
        self.about_me_collection = self.chroma_client.get_or_create_collection(
            name="about_me",
            embedding_function=self.embedding_function
        )
        
        # Load and embed JSON data
        self.load_json_data(json_path)
    
    def load_json_data(self, json_path: str):
        """
        Load JSON data and embed it into ChromaDB
        :param json_path: Path to the JSON file
        """
        # Read JSON file
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except FileNotFoundError:
            print(f"Error: File {json_path} not found.")
            return
        except json.JSONDecodeError:
            print(f"Error: Invalid JSON format in {json_path}.")
            return
        
        # Prepare data for embedding
        documents = []
        metadatas = []
        ids = []
        
        # Process topics from the JSON structure
        for idx, topic in enumerate(data.get('topics', [])):
            # Create a comprehensive text document from the topic
            document_parts = [
                f"Title: {topic.get('title', '')}",
                f"Description: {topic.get('description', '')}"
            ]
            
            # Add key competences if present
            if 'key_competences' in topic:
                competences = '\n'.join([f"- {comp}" for comp in topic['key_competences']])
                document_parts.append(f"Key Competences:\n{competences}")
                
            # Add major providers if present
            if 'keywords' in topic:
                providers = '\n'.join([f"- {provider}" for provider in topic['keywords']])
                document_parts.append(f"Keywords:\n{providers}")
                
            # Join all parts into a single document
            document = '\n\n'.join(document_parts)
            documents.append(document)
            
            # Store metadata
            metadatas.append({
                "source": json_path,
                "index": idx,
                "title": topic.get('title', '')
            })
            ids.append(f"topic_{idx}")
        
        # Add documents to ChromaDB collection
        if documents:
            self.about_me_collection.add(
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )
            print(f"Loaded {len(documents)} topics into ChromaDB")
        else:
            print("No topics found in the JSON file.")
    
    
    def retrieve_context(self, query: str, top_k: int = 3) -> List[str]:
        """
        Retrieve most relevant context for a given query
        
        :param query: User's query
        :param top_k: Number of top results to retrieve
        :return: List of retrieved context documents
        """
        # Retrieve top K most similar documents
        results = self.about_me_collection.query(
            query_texts=[query],
            n_results=top_k
        )
        
        # Extract and parse documents
        contexts = []
        for doc in results['documents'][0]:
            contexts.append(doc)
        
        return contexts
    
 
    def generate_response(self, query: str) -> str:
        """
        Generate a response using retrieved context and OpenAI
        
        :param query: User's query
        :return: Generated response
        """
        # Retrieve context
        contexts = self.retrieve_context(query)
        
        # Prepare prompt with context
        prompt = f"""
        Knowledge base: {' '.join(contexts)}
        
        Question: {query}
        
        Based on the provided context, please answer the question thoroughly and precisely.
        If the context  isn't realated to Luděk kvapil, his life, education, experiences and another information
        stored in knowledge base, respond that you are sorry, but you can answer only topics about Luděk and his work.
        """
        
        try:
            # Generate response using OpenAI
            completion = client.chat.completions.create(
                model=MODEL, 
                messages=[
                    {"role": "system", "content": "You are a personal assistant who respond questions about Luděk Kvapil."},
                    {"role": "user", "content": prompt}
                ],
            )

            return completion.choices[0].message.content
        except Exception as e:
            return f"An error occurred while generating the response: {str(e)}"

def main():
    # Example usage
    rag_app = RAGApplication('knowledge_base.json')
    
    # Example queries
    queries = [
        
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        response = rag_app.generate_response(query)
        print(f"Response: {response}")

if __name__ == '__main__':
    main()

Loaded 3 topics into ChromaDB

Query: What is Luděk's education?
Response: Luděk Kvapil studied sociology, adult education, and andragogy at Palacky University (UPOL). His bachelor's thesis focused on social movement mobilization, where he attempted to validate the theory of John McCarthy and Mayer Zald regarding the effectiveness of promoting the NGO 'Hnutí Duha' program in the context of media reporting. His educational background also includes knowledge in statistics, psychology, philosophy, science methodology, and opinion polls.


In [22]:
rag_app = RAGApplication('knowledge_base.json')

Insert of existing embedding ID: topic_0
Insert of existing embedding ID: topic_1
Insert of existing embedding ID: topic_2
Insert of existing embedding ID: topic_3
Insert of existing embedding ID: topic_4
Insert of existing embedding ID: topic_5
Insert of existing embedding ID: topic_6
Insert of existing embedding ID: topic_7
Insert of existing embedding ID: topic_8
Add of existing embedding ID: topic_0
Add of existing embedding ID: topic_1
Add of existing embedding ID: topic_2
Add of existing embedding ID: topic_3
Add of existing embedding ID: topic_4
Add of existing embedding ID: topic_5
Add of existing embedding ID: topic_6
Add of existing embedding ID: topic_7
Add of existing embedding ID: topic_8


Loaded 9 topics into ChromaDB


In [13]:
# Retrieve all documents from the collection
all_documents = rag_app.about_me_collection.get()

all_documents

{'ids': ['topic_0',
  'topic_1',
  'topic_2',
  'topic_3',
  'topic_4',
  'topic_5',
  'topic_6',
  'topic_7',
  'topic_8'],
 'embeddings': None,
 'documents': ['Title: Who is Luděk Kvapil?\n\nDescription: Luděk is Drupal backend developer with experience in PHP, AWS, OpenTofu/Terraform, GitLab CI/CD, and more.\n\nKey Competences:\n- More that 13 years experience with Drupal\n- Development of custom an contrib modules\n- LLM integrations and RAG\n- AWS Developer associate\n- ECS, EC2, RDS, S3, Lambda, EventBridge, StepFunctions, Redis, OpenSearch, CloudFront, IAM, SecretManager\n- API integration\n- Docker images development\n- CI/CD pipelines\n- CloudFormation, Terraform/OpenTofu',
  "Title: What is a Drupal backend developer?\n\nDescription: A Drupal backend developer builds and maintains server-side functionality in Drupal websites. They create custom modules, integrate APIs, manage databases, and optimize performance. They work with PHP, Symfony, and Drupal's hook and service syste

In [24]:
# question = "What is Luděk kvalifications?"

question = "Who is Luděk Kvapil?"

In [25]:
context = rag_app.retrieve_context(question)

context

['Title: Who is Luděk Kvapil?\n\nDescription: Luděk is Drupal backend developer with experience in PHP, AWS, OpenTofu/Terraform, GitLab CI/CD, and more.\n\nKey Competences:\n- More that 13 years experience with Drupal\n- Development of custom an contrib modules\n- LLM integrations and RAG\n- AWS Developer associate\n- ECS, EC2, RDS, S3, Lambda, EventBridge, StepFunctions, Redis, OpenSearch, CloudFront, IAM, SecretManager\n- API integration\n- Docker images development\n- CI/CD pipelines\n- CloudFormation, Terraform/OpenTofu',
 "Title: What Luděk studied in University Palacky (UPOL)? What is Luděk's education?\n\nDescription: His bachelor's thesis was on the topic of social movement mobilization, where he tried to validate the theory of John McCarthy and Mayer Zald on the effectiveness of promoting the NGO 'Hnutí Duha' program in the context of the number of reports in the media.\n\nKeywords:\n- Sociology\n- Adult education\n- Andragogy\n- Statistic\n- Psychology\n- Philosophy\n- Scienc

In [19]:
response = rag_app.generate_response(question)
response

"Luděk Kvapil's qualifications include over 13 years of experience as a Drupal backend developer, where he has honed his skills in various areas, including:\n\n- Development of custom and contributed Drupal modules.\n- Expertise in PHP and API integration.\n- Proficiency in AWS services such as ECS, EC2, RDS, S3, Lambda, EventBridge, Step Functions, Redis, OpenSearch, CloudFront, IAM, and SecretManager.\n- Knowledge in Docker images development and CI/CD pipelines using GitLab CI/CD.\n- Familiarity with infrastructure as code tools like CloudFormation and Terraform/OpenTofu.\n- Experience with LLM integrations and RAG (retrieval-augmented generation).\n\nAdditionally, Luděk has a bachelor's degree in Sociology from Palacky University (UPOL), with a thesis focused on social movement mobilization, and has studied various related subjects such as adult education, statistics, psychology, philosophy, and science methodology.\n\nOverall, his qualifications reflect a blend of technical expert